# From notebook exploration to a tested Python package

Refactor Lecture 2's scientific knowledge graph into reusable modules, a deliberate public API
(the supported names and behavior other code may rely on),
and a thin command-line application.

**Lecture 3 · Notebook 01 · CMOR 438 / INDE 577**

## Orientation

**Core:** artifact roles, module execution and caching, package structure, public interfaces,
`__main__`, command-line adapters, and tests.

**Practice:** classify code by responsibility and inspect a package without hard-coding its files.

**Extension:** evolve the CLI without coupling presentation to domain behavior.

Notebook 00 established the environment. This notebook uses that environment to turn the working
knowledge-graph prototype from Lecture 2 into maintainable software.

## How to use this notebook

**Estimated time:** 60 minutes core, plus 35 minutes of practice and extension.

**Prerequisites:** the Rice DSM kernel, functions, exceptions, dataclasses, native files, and the
knowledge-graph lesson.

Run top to bottom. Predict which process executes each example and which names it creates. The
notebook reads the actual package, runs temporary scripts in isolated directories, and invokes the
package CLI in subprocesses. It does not modify source code while executing.

## Learning objectives

By the end, you should be able to:

- distinguish a notebook, script, module, import package, and distribution package;
- decide what belongs in exploratory analysis, reusable library code, or an application adapter;
- explain module namespaces, first-import execution, `sys.modules`, and stale kernel state;
- use `__name__ == "__main__"` and package `__main__.py` appropriately;
- explain why import-time I/O, printing, network calls, and argument parsing are dangerous;
- design a small public package interface without confusing `__all__` with access control;
- keep computation separate from command-line parsing, output, and exit status;
- invoke a package portably with `python -m`; and
- use unit and subprocess tests to protect both library and CLI contracts.

## Why this matters in industry and science

An exploratory notebook is excellent for asking questions. It becomes a fragile integration point
when multiple notebooks copy the same parser, model, or algorithm. Fixes diverge, hidden state grows,
and results depend on which version of a cell someone last ran.

Reusable scientific software needs stable boundaries: library functions return values or raise
documented exceptions; applications translate user input into library calls; notebooks retain the
questions, interpretation, and visual narrative. Packaging makes those boundaries installable and
testable—it does not automatically make the design good.

## Worked example: promote the scientific knowledge graph

Lecture 2 built nodes, relationships, a directed graph, JSON/CSV loaders, and breadth-first search in
one notebook. The definitions are now useful beyond that narrative, so Lecture 3 promotes them:

```text
notebook 03                         rice_dsm package
-------------------------           --------------------------------
exploration and explanation    →    knowledge_graph.py: domain + I/O
print-oriented interaction     →    cli.py: arguments + presentation
Run All                        →    tests: executable behavior contracts
one notebook entry             →    __main__.py + console entry point
```

The original notebook remains self-contained teaching history. The package becomes the maintained
implementation used by later work.

## Professional practice

| Data scientist asks | Software engineer asks |
| --- | --- |
| Which parts express the scientific question? | Which parts form reusable interfaces? |
| Is a transformation valid for these data? | Where is its invariant enforced and tested? |
| Does output preserve uncertainty and provenance? | Does presentation remain outside computation? |
| Can another analysis reuse the method? | Can it import the method without side effects? |
| Is a graph path interpreted honestly? | Is traversal deterministic and documented? |
| What changed between results? | Which module, version, and tests changed? |

Modularity serves scientific review when it makes assumptions and transformations easier to inspect.

In [ ]:
import importlib
import importlib.metadata
import inspect
import pkgutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

## 1. Artifact names describe roles, not just suffixes

| Artifact | Primary role |
| --- | --- |
| notebook | interactive narrative containing code, explanation, and output |
| script | `.py` file used as a program's top-level entry |
| module | one importable namespace, commonly backed by one `.py` file |
| regular package | importable module with submodules and usually `__init__.py` |
| distribution package | installable project with metadata, such as `rice-dsm` |

One file can serve more than one role. Executing `tool.py` treats it as top-level script code;
importing `tool` treats it as a module. That flexibility requires deliberate boundaries.

### The package lives under `src/`

The source layout prevents the repository root from accidentally masquerading as an installed
package. Code must pass through the packaging configuration and editable installation before
`import rice_dsm` works reliably.

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest Rice DSM project directory.

    Parameters
    ----------
    start : Path
        Directory from which to search upward.

    Returns
    -------
    Path
        Directory containing this course's project declaration.

    Raises
    ------
    FileNotFoundError
        If the project cannot be found.
    """

    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        project_file = candidate / "pyproject.toml"
        if project_file.is_file() and "rice-dsm" in project_file.read_text(
            encoding="utf-8"
        ):
            return candidate
    raise FileNotFoundError(f"rice-dsm project not found above {resolved_start}")


project_root = find_project_root(Path.cwd())
package_directory = project_root / "src" / "rice_dsm"
package_files = tuple(sorted(path.name for path in package_directory.glob("*.py")))

print(package_files)
assert {
    "__init__.py",
    "__main__.py",
    "cli.py",
    "knowledge_graph.py",
    "records.py",
} <= set(package_files)

## 2. Importing creates and initializes a module object

An import does more than paste definitions into a notebook:

1. find a module specification using the import system;
2. create a module object and namespace;
3. place it in `sys.modules`;
4. execute its top-level statements to initialize that namespace; and
5. bind a name in the importer.

Top-level function and class definitions are statements too: executing them creates those objects.
Top-level side effects—file reads, printing, network calls, or model training—would also execute,
which is why import-time work must remain cheap and predictable.

In [ ]:
knowledge_graph_module = importlib.import_module("rice_dsm.knowledge_graph")

print("Module object: ", knowledge_graph_module)
print("__name__:      ", knowledge_graph_module.__name__)
print("__package__:   ", knowledge_graph_module.__package__)
print("__file__:      ", knowledge_graph_module.__file__)
print("Loader:        ", type(knowledge_graph_module.__spec__.loader).__name__)

assert knowledge_graph_module.__name__ == "rice_dsm.knowledge_graph"
assert Path(knowledge_graph_module.__file__).resolve().is_relative_to(
    package_directory.resolve()
)

### A module owns a namespace

`import rice_dsm.knowledge_graph` binds the package name; an alias can bind the module directly.
Attributes remain qualified, making their origin visible. `from module import name` binds selected
objects directly in the current namespace. Neither style copies the function implementation.

In [ ]:
public_module_names = tuple(
    sorted(name for name in vars(knowledge_graph_module) if not name.startswith("_"))
)
print(public_module_names)

assert "KnowledgeGraph" in public_module_names
assert "load_knowledge_graph" in public_module_names
assert "_require_string" not in public_module_names

## 3. Imports are cached per interpreter process

For a given fully qualified name, a normal second import returns the object already stored in
`sys.modules`; it does not re-execute the file. This prevents repeated initialization but surprises
students who edit source while a kernel remains alive.

In [ ]:
first_module_identity = id(knowledge_graph_module)
second_import = importlib.import_module("rice_dsm.knowledge_graph")

print("First identity: ", first_module_identity)
print("Second identity:", id(second_import))
print("Cached object:  ", sys.modules["rice_dsm.knowledge_graph"])

assert second_import is knowledge_graph_module
assert sys.modules["rice_dsm.knowledge_graph"] is knowledge_graph_module

### Common failure: reaching for `reload` too quickly

`importlib.reload(module)` re-executes module code but does not rebuild every object already imported
elsewhere. Existing instances still belong to old class objects; dependent modules may still hold old
function references. For structural package changes, restart the kernel and Run All. Treat reload as
an interactive convenience, not proof that a clean process works.

## 4. Refactor by responsibility, not file length

The promoted implementation uses three layers:

- `knowledge_graph.py`: domain objects, invariants, traversal, and native-file adapters;
- `cli.py`: command-line parsing, user-facing text, and exit-status policy;
- `__main__.py`: minimal package execution hook.

File count is not the objective. A boundary is useful when it produces a coherent reason to change
and allows the inner behavior to be called without the outer interface.

In [ ]:
from rice_dsm.knowledge_graph import KnowledgeGraph, load_knowledge_graph

print(inspect.signature(load_knowledge_graph))
print(inspect.getdoc(KnowledgeGraph.find_path).splitlines()[0])

assert inspect.signature(load_knowledge_graph).return_annotation == "KnowledgeGraph"
assert "breadth-first" in inspect.getdoc(KnowledgeGraph.find_path).lower()

### What remains in the notebook?

Keep the scientific question, hypotheses, plots, intermediate inspection, alternative approaches,
and interpretation close to the reader. Move code when it becomes shared behavior, needs isolated
tests, represents a stable domain concept, or serves an application interface.

“Move everything out of notebooks” is not a professional rule. The goal is a thin narrative calling
well-tested computational units—not a notebook stripped of reasoning.

## 5. Build a deliberate package-level API

Here **API** means application programming interface: the supported names and behaviors that other
Python code may rely on. This is an in-process library API, not an HTTP web service. Both kinds are
contracts, but a library call exchanges Python objects directly while a web API exchanges requests
and responses across a process or network boundary.

`rice_dsm/__init__.py` re-exports selected names so users can write:

```python
from rice_dsm import KnowledgeGraph, load_knowledge_graph
```

instead of depending on internal file placement. This facade can remain stable if implementations
move. Every exported name becomes a compatibility promise that requires documentation and tests.

In [ ]:
import rice_dsm

print(rice_dsm.__all__)
expected_public_names = {
    "KnowledgeGraph",
    "KnowledgeNode",
    "Relationship",
    "StudentRecord",
    "load_knowledge_graph",
    "mean_absolute_error",
    "root_mean_squared_error",
    "summarize_scores",
}

assert set(rice_dsm.__all__) == expected_public_names
assert rice_dsm.KnowledgeGraph is KnowledgeGraph

### `__all__` and leading underscores are conventions, not security

`__all__` controls wildcard-import behavior and documents our intended facade. A leading underscore
marks an implementation detail by convention. Python still permits an informed caller to access
those names.

Avoid `from package import *` in maintained code: it obscures name origins and can change behavior
when the package exports a new name. Import the module or explicit public names instead.

### Internal imports: absolute or relative?

Inside `rice_dsm`, both can express package relationships:

```python
from rice_dsm.knowledge_graph import load_knowledge_graph  # explicit absolute import
from .knowledge_graph import load_knowledge_graph          # explicit relative import
```

This repository favors absolute imports because their full origin is visible. That is a project
convention, not a language requirement. Be consistent and avoid dependency cycles either way.

### Common failure: circular imports

If `cli.py` imports `knowledge_graph.py` while `knowledge_graph.py` imports `cli.py`, Python can expose
a partially initialized module. The better repair is usually architectural: dependencies should flow
from outer adapters toward inner domain logic.

```text
__main__.py → cli.py → knowledge_graph.py
```

The domain module must not know how the terminal presents it.

## 6. Script execution changes `__name__`

When imported, a module's `__name__` is its import name. When used as the top-level program, Python
sets `__name__` to `"__main__"`. A guard can keep application behavior from running during import:

```python
def main() -> int:
    ...
    return 0

if __name__ == "__main__":
    raise SystemExit(main())
```

Keep the guarded block minimal so `main` remains directly testable.

### Controlled experiment: import the same file and run it

The experiment creates a temporary module with one guarded print. Two fresh subprocesses prevent the
notebook kernel's module cache from affecting the comparison. Argument lists avoid shell-specific
quoting on Windows, macOS, and Linux.

In [ ]:
temporary_module_source = """def identity() -> str:
    return __name__

if __name__ == "__main__":
    print("executed-as", identity())
"""

with TemporaryDirectory() as temporary_directory:
    temporary_path = Path(temporary_directory)
    module_path = temporary_path / "boundary_demo.py"
    module_path.write_text(temporary_module_source, encoding="utf-8")

    script_result = subprocess.run(
        [sys.executable, str(module_path)],
        check=True,
        capture_output=True,
        text=True,
    )
    import_result = subprocess.run(
        [
            sys.executable,
            "-c",
            "import boundary_demo; print(boundary_demo.identity())",
        ],
        cwd=temporary_path,
        check=True,
        capture_output=True,
        text=True,
    )

print("Run as file:", script_result.stdout.strip())
print("Imported:   ", import_result.stdout.strip())
assert script_result.stdout.strip() == "executed-as __main__"
assert import_result.stdout.strip() == "boundary_demo"

## 7. A package can be an application with `__main__.py`

`python -m rice_dsm` asks the import system to locate the installed package, then executes
`rice_dsm/__main__.py` as the top-level environment. Our `__main__.py` contains only delegation:

```python
from rice_dsm.cli import main

raise SystemExit(main())
```

The testable behavior remains in `cli.main`; the hook owns no domain logic.

### Library versus CLI contract

| Library (`knowledge_graph.py`) | CLI (`cli.py`) |
| --- | --- |
| accepts Python objects | parses argument strings |
| returns objects or values | formats text for a person |
| raises specific exceptions | translates failures into messages and status codes |
| avoids printing | writes to stdout or stderr |
| reusable in notebooks and services | represents one terminal interface |

Printing inside `find_path` would couple every caller to terminal presentation. Returning a tuple
lets a notebook, API, test, or CLI choose its own representation.

In [ ]:
data_directory = (
    project_root / "notebooks" / "lecture-02-python-foundations-ii" / "data"
)
concepts_path = data_directory / "scientific_concepts.json"
relationships_path = data_directory / "scientific_relationships.csv"

graph = load_knowledge_graph(concepts_path, relationships_path)
path = graph.find_path("diffusion_model", "scientific_measurement")

assert path is not None
print(path)
assert path[0] == "diffusion_model"
assert path[-1] == "scientific_measurement"

### Invoke the package portably

Use the active interpreter rather than assuming a platform-specific executable path. The first
subprocess requests the default summary; the second sends a path query through the complete CLI.

In [ ]:
base_command = [
    sys.executable,
    "-m",
    "rice_dsm",
    str(concepts_path),
    str(relationships_path),
]
summary_process = subprocess.run(
    base_command,
    check=True,
    capture_output=True,
    text=True,
)
path_process = subprocess.run(
    [
        *base_command,
        "--path",
        "diffusion_model",
        "scientific_measurement",
    ],
    check=True,
    capture_output=True,
    text=True,
)

print("Summary:", summary_process.stdout.strip())
print("Path:   ", path_process.stdout.strip())
assert summary_process.stdout.strip() == "20 nodes; 25 relationships"
assert path_process.stdout.startswith("Diffusion model -> Markov chain")

### Console entry points add a user-facing command

`[project.scripts]` in `pyproject.toml` maps `rice-dsm` to `rice_dsm.cli:main`. Installation creates a
small platform-appropriate launcher. The console command and `python -m rice_dsm` reach the same
function; `python -m` is especially explicit about which interpreter supplies the package.

In [ ]:
from rice_dsm.cli import main

distribution = importlib.metadata.distribution("rice-dsm")
console_entries = tuple(
    entry
    for entry in distribution.entry_points
    if entry.group == "console_scripts" and entry.name == "rice-dsm"
)

assert len(console_entries) == 1
console_entry = console_entries[0]
print("Entry point:", console_entry)
print("Loads:      ", console_entry.load())

assert console_entry.load() is main

### Exit status is part of an application contract

Conventionally, zero means success and nonzero signals a failure or unmet request. Our CLI returns:

- `0`: graph loaded and requested output produced;
- `1`: valid graph, but no directed path exists; and
- `2`: unreadable/invalid input or an invalid node request.

The library still uses exceptions and `None` according to its Python interfaces. Translation happens
only at the application boundary.

In [ ]:
unreachable_process = subprocess.run(
    [
        *base_command,
        "--path",
        "scientific_measurement",
        "diffusion_model",
    ],
    check=False,
    capture_output=True,
    text=True,
)

print("Status:", unreachable_process.returncode)
print("Output:", unreachable_process.stdout.strip())
assert unreachable_process.returncode == 1
assert "no directed path" in unreachable_process.stdout

## 8. Tests replace notebook memory with repeatable contracts

Package tests start from imports and construct fresh objects. They protect:

- value-object validation;
- duplicate node and relationship rejection;
- referential integrity;
- predicate-filtered neighbors;
- shortest directed path behavior;
- JSON/CSV integration; and
- CLI output, stderr, and exit status.

Notebook assertions explain one narrative. Unit tests systematically protect behavior as the package
changes. Both run in CI on Windows, macOS, and Linux.

In [ ]:
test_file = project_root / "tests" / "test_knowledge_graph.py"
test_source = test_file.read_text(encoding="utf-8")

protected_behaviors = (
    "duplicate_nodes",
    "unknown_endpoints",
    "breadth_first_search",
    "native_file_loader",
    "cli_prints_summary_and_path",
    "cli_returns_nonzero",
)
for behavior in protected_behaviors:
    print(f"{behavior:<30}", behavior in test_source)

assert all(behavior in test_source for behavior in protected_behaviors)

## Debugging imports and applications systematically

1. **Environment:** does `sys.prefix` identify the intended `.venv`?
2. **Specification:** what does `importlib.util.find_spec(full_name)` report?
3. **Origin:** does `module.__file__` point to the intended checkout?
4. **Cache:** is the name already in `sys.modules` from an older import?
5. **Dependency direction:** is a circular import exposing partial initialization?
6. **Execution role:** is a file imported, run by path, or invoked with `-m`?
7. **Arguments:** does the CLI receive the expected strings and working directory?
8. **Failure channel:** is the message on stderr and the status nonzero?
9. **Clean process:** does the same test pass in a fresh subprocess and CI?

Do not rename files, alter `sys.path`, or add imports until the broken boundary is identified.

### Common failure modes

| Failure | Why it happens | Design response |
| --- | --- | --- |
| local `json.py` shadows standard library | current directory is on search path | choose non-conflicting module names |
| importing starts a workflow | top-level side effect | move execution into `main` |
| edited function appears unchanged | module cached in kernel | restart and Run All |
| relative data path fails | process has another working directory | accept a `Path` from the caller |
| circular import error | modules depend both directions | extract inner domain boundary |
| notebook imports private helper | caller couples to implementation | promote or replace with public API |
| CLI test calls a shell string | quoting differs by platform | pass an argument list |
| error prints but status is zero | application contract incomplete | return deliberate status code |

## Guided practice: classify responsibilities before moving code

For each item, choose `notebook`, `domain module`, `I/O adapter`, or `CLI adapter`, then explain the
reason to change:

1. prose interpreting why a graph path is scientifically limited;
2. validation that confidence lies in `[0, 1]`;
3. CSV field conversion with line-number context;
4. terminal option `--path START END`;
5. a plot comparing path lengths across graph versions; and
6. breadth-first search.

Success criterion: every placement names its callers, outputs, failure policy, and likely tests.

In [ ]:
responsibility_map = {
    "scientific path interpretation": "notebook",
    "confidence invariant": "domain module",
    "CSV conversion": "I/O adapter",
    "--path option": "CLI adapter",
    "comparative plot": "notebook",
    "breadth-first search": "domain module",
}

for responsibility, destination in responsibility_map.items():
    print(f"{destination:<15} | {responsibility}")

assert set(responsibility_map.values()) == {
    "notebook",
    "domain module",
    "I/O adapter",
    "CLI adapter",
}

## Independent practice: inventory an installed package

Implement `module_inventory(package_name)` using `importlib.import_module` and
`pkgutil.iter_modules`. Return sorted fully qualified module names.

Success criteria: type hints, NumPy-style docstring, a useful error for a non-package module, no
hard-coded package files, and assertions that discover `rice_dsm.cli` and
`rice_dsm.knowledge_graph`.

In [ ]:
def module_inventory(package_name: str) -> tuple[str, ...]:
    """Return immediate importable modules contained in a regular package.

    Parameters
    ----------
    package_name : str
        Fully qualified package to inspect.

    Returns
    -------
    tuple of str
        Sorted fully qualified names of immediate child modules.

    Raises
    ------
    ValueError
        If the imported object is not a package with ``__path__``.
    """

    package = importlib.import_module(package_name)
    package_paths = getattr(package, "__path__", None)
    if package_paths is None:
        raise ValueError(f"{package_name!r} is a module, not a package")
    return tuple(sorted(
        module_info.name
        for module_info in pkgutil.iter_modules(
            package_paths,
            prefix=f"{package_name}.",
        )
    ))


rice_dsm_modules = module_inventory("rice_dsm")
print(rice_dsm_modules)
assert "rice_dsm.cli" in rice_dsm_modules
assert "rice_dsm.knowledge_graph" in rice_dsm_modules

## Extension: evolve the CLI without contaminating the library

Design a `--format text|json` option. JSON output is useful for automation, but raises choices:

- What schema version identifies the output contract?
- Should labels, identifiers, predicates, or complete relationship evidence be emitted?
- Does “no path” produce valid JSON with status `1`, or a successful empty result?
- Which messages belong on stderr?
- How will a test prevent accidental prose from corrupting machine-readable stdout?

Keep `KnowledgeGraph.find_path` unchanged. The same returned tuple should support both presentations.
That constraint tests whether the library/application separation is genuine.

## Retrieval practice

Answer without running code:

1. How can the same `.py` file act as a script or a module?
2. What happens during the first import, and what does `sys.modules` change later?
3. Why can `reload` leave a notebook with mixed old and new objects?
4. What belongs in the knowledge-graph module versus its CLI?
5. What does `__all__` communicate, and what does it not prevent?
6. Why is `__main__.py` deliberately tiny?
7. How do `python -m rice_dsm` and the `rice-dsm` entry point converge?
8. Why should subprocess calls pass an argument list rather than a shell command string?
9. Which parts of scientific reasoning should remain in a notebook?

## Takeaway

A professional scientific Python workflow separates roles:

```text
notebook question and interpretation
        ↓ calls
package domain behavior and I/O
        ↑ protected by tests
CLI adapter and other applications
```

Modules create namespaces and reusable boundaries; packages organize and publish those boundaries;
scripts and entry points initiate applications. Imports should initialize definitions cheaply and
predictably—not start the analysis.

Next, Notebook 02 introduces NumPy arrays as the package's first major numerical dependency.

## Further reading

- [Python tutorial: Modules and packages](https://docs.python.org/3/tutorial/modules.html)
- [Python: `__main__` and package `__main__.py`](https://docs.python.org/3/library/__main__.html)
- [Python: The import system](https://docs.python.org/3/reference/import.html)
- [Python: `importlib`](https://docs.python.org/3/library/importlib.html)
- [Python Packaging User Guide: src layout versus flat layout](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/)
- [Python Packaging User Guide: Creating command-line tools](https://packaging.python.org/en/latest/guides/creating-command-line-tools/)
- [Python: `argparse`](https://docs.python.org/3/library/argparse.html)